In [1]:
import sys
import scanpy as sc
from train import manual_train_fm
import os
import json
import torch
import random
from data_loader import create_dataloader_eval

# Configurate the data

In [14]:
datapath = "/scratch/project_465001027/Spatialformer/cache/xenium_THD0008_pair"
# datapath = "/scratch/project_465001027/Spatialformer/cache/xenium_TILD175_pair"
dataloader = create_dataloader_eval(datapath, 
                                    num_workers = 4, 
                                    batch_size = 4,
                                    # batch_size = 16,
                                    directionality = True,
                                    context_length = 500, 
                                    padding_idx = 0, 
                                    special_token_num = 4, 
                                    n_bins = 51, 
                                    sep_token = 1949, 
                                    cls_token = 1)
get_file_path = lambda path, filename: os.path.join("/scratch/project_465001027/Spatialformer", path, filename)
config_path = get_file_path("config", "_config_train_large_pair.json")
with open(config_path, 'r') as json_file:
    config = json.load(json_file)
model_ckp_path = "/scratch/project_465001027/Spatialformer/output/checkpoints/step=0025000-train_total_loss=-1.3222-val_total_loss=0.0000.ckpt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Configurate the model

In [15]:
model = manual_train_fm(config = config)
ckp = torch.load(model_ckp_path, map_location=torch.device(device))
params = ckp["state_dict"]
model.load_state_dict(params)
model.eval()
model.to(device)

[rank: 0] Global seed set to 42


require grad: True


Spaformer(
  (encoder): SpaEncoder(
    (emb_proj): Linear(in_features=2560, out_features=512, bias=True)
    (emb_dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (bpp_feature_proj): Linear(in_features=3, out_features=512, bias=True)
    (bpp_convnet): BPPConvnet(
      (conv1): MaskedConv2d(
        (conv): Conv2dSame(
          (conv): Conv2d(1, 64, kernel_size=(7, 7), stride=(1, 1))
        )
      )
      (act1): ReLU()
      (conv2): MaskedConv2d(
        (conv): Conv2dSame(
          (conv): Conv2d(64, 64, kernel_size=(7, 7), stride=(1, 1))
        )
      )
      (final_act): Identity()
    )
    (layers): ModuleList(
      (0-9): 10 x SqueezeformerBlock(
        (conv_blocks): ModuleList(
          (0): Conv1DBlock(
            (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (glu): GLU(
              (activation): Si

In [28]:
def rearrange_sentences(batch):
    # Extract the relevant tensors from the batch dictionary
    batch_size = len(batch['indices'])
    max_length = 500
    indices = batch['indices']
    mask_attentions = batch['attention_mask']  # Corrected to match your input
    token_types = batch['token_type_ids']  # Corrected field names
    pair_label = batch["pair_label"]
    new_indices = torch.full((batch_size, max_length), 0, dtype=torch.int)
    new_mask_attention = torch.full((batch_size, max_length), 0, dtype=torch.int)
    new_token_types = torch.full((batch_size, max_length), 0, dtype=torch.int)

    sep_token = 1949  # This is the token ID for the separator

    for i, (indice, mask_attention, token_type) in enumerate(zip(indices, mask_attentions, token_types)):
        # Find the index of the separator token (1949)
#         print("batch_size", batch_size)
        mid_index = (indice == sep_token).nonzero(as_tuple=True)[0][0]
        end_index = (indice == sep_token).nonzero(as_tuple=True)[0][1]
        if mid_index.numel() == 0:  # Check if SEP token is present
            print(f"Warning: SEP token not found in batch index {i}.")
            continue  # Skip to the next batch if no SEP token is found
        
        # Split the indices into two sentences
        cls = indice[:1]  # Include the CLS token
        sentence1 = indice[1:mid_index + 1]  # From index 1 to the SEP token (inclusive)
        sentence2 = indice[mid_index + 1:end_index+1]   # From the token after the SEP to the end
        # Combine to form the new order: sentence2 followed by sentence1
        combined = torch.cat((cls, sentence2, sentence1))
        
        # Pad to the max_length
        new_sequence = torch.cat((combined, torch.zeros(max_length - combined.size(0), dtype=torch.int)))
#         print(new_sequence)
        # Update new_indices
        new_indices[i, :] = new_sequence[:max_length]  # Ensure to only keep the first max_length tokens

        # Update the attention mask
        new_mask_attention[i, :] = mask_attention  

        # Update the token type IDs
        left_token_type = torch.full((len(cls)+len(sentence2),), 1)  # Token type for the second sentence
        right_token_type = torch.full((len(sentence1),), 2)  # Token type for the first sentence
        pad_token_type = torch.full((max_length - combined.size(0),), 0)  # Padding type
        # Combine token types
        new_token_type = torch.cat((left_token_type, right_token_type, pad_token_type))

        # Update new_token_types
        new_token_types[i, :] = new_token_type[:max_length]

    # Put it all back in a dictionary
    new_batch = {
        "indices": new_indices,  # Add batch dimension
        "attention_mask": new_mask_attention.bool(),  # Add batch dimension
        "token_type_ids": new_token_types,  # Add batch dimension
        "pair_label": pair_label
    }
    
    return new_batch

def get_acc(pred, target):
    target = target.to(device)
    # import pdb; pdb.set_trace()
    _, pred = torch.max(pred.data, 1)  # Get the predicted class
    if (pred == 0).any():
        print("negative prediction:", pred)
    total = target.size(0)  # Total number of samples
    correct = (pred == target).sum().item()  # Count correct predictions
    accuracy = correct / total
    return accuracy

def process_tensors(tensor1, tensor2):
    # Ensure inputs are tensors
    if not isinstance(tensor1, torch.Tensor) or not isinstance(tensor2, torch.Tensor):
        raise ValueError("Both inputs must be PyTorch tensors.")

    # Verify that both tensors have exactly two elements
    if tensor1.numel() != 2 or tensor2.numel() != 2:
        raise ValueError("Both tensors must contain exactly two elements.")

    # Check if both tensors have the order element2 > element1
    are_tensor1_aligned = tensor1[1] > tensor1[0]
    are_tensor2_aligned = tensor2[1] > tensor2[0]

    # Process based on the conditions
    if are_tensor1_aligned and are_tensor2_aligned:
        # Both tensors are aligned, so randomly choose one
        chosen_tensor = random.choice([tensor1, tensor2])
    else:
        # Neither tensor is aligned, choose based on the first element
        if tensor1[0] > tensor1[1]:
            chosen_tensor = tensor1
        elif tensor2[0] > tensor2[1]:
            chosen_tensor = tensor2

    return chosen_tensor

# Example usage:
tensor1 = torch.tensor([0.34, 0.59])
tensor2 = torch.tensor([0.75, 0.89])

result_tensor = process_tensors(tensor1, tensor2)
print("Selected Tensor:", result_tensor)

Selected Tensor: tensor([0.3400, 0.5900])


In [30]:
from tqdm import tqdm
counter = 0
pos_counter = 0
neg_counter = 0
predicted = []
label = []
with torch.no_grad():
    for i, batch in tqdm(enumerate(dataloader)):
        if i<10000:
            pair_label = batch["pair_label"]
#             print("pair_label:", pair_label)
            label.append(pair_label)
            counter += len(pair_label)
            pos_counter += (pair_label == 1).sum().item()
            neg_counter += (pair_label == 0).sum().item()
            #order 1
            last_hidden_repr, co_adj_prob = model.get_embeddings(batch, [-1], True, False)
            #reverse order 
            reorder_batch = rearrange_sentences(batch)
            rev_last_hidden_repr, rev_adj_prob = model.get_embeddings(reorder_batch, [-1], True, False)
            result_tensors = list(map(lambda x, y: process_tensors(x, y), co_adj_prob, rev_adj_prob))
            result_tensor = torch.stack(result_tensors)
            predicted.append(result_tensor)
#             print(predicted)
        else:
            break

predicts = torch.concat(predicted)
targets = torch.concat(label)
acc = get_acc(predicts, targets)
print(f"accuracy of the pair prediction: {acc}")
print(f"There are {counter} samples that are token into account")
print(f"{pos_counter} is positive; {neg_counter} is negative")

10000it [36:08,  4.61it/s]


accuracy of the pair prediction: 0.5
There are 40000 samples that are token into account
20000 is positive; 20000 is negative


<span style="color: red; font-size: 20px;">&#10071;</span> This result shows that our model cannot capture the cross slide cellular structure, which means that the cell cannot have a stable niche environment across different slides. The same evidence can also be proved by the Lung samples of David's data.

# We have two tasks for validating the spatial information we encode previously.

1) SC-ST (Single cell - Spatial transcriptomics). The cellular spatial expression patterns should be robust after training in a large scale of spatial transcriptomics dataset. Even if the cell type without spatial co-localization patterns can be espected, thoes cell types with strong spatial cell-cell colocalization patterns should be revealed. In this case, if we input the single-cell expression profile, the spatial co-localization information should be retrived directly if there exist this information. This is extremely helpful when only having single-cell dataset but still want to get the spatial co-localization patterns. Instead of testing the cross cell-type co-localization, one can also apply the model to get the inner cell-type distribution, indicating the spatial scarcity of a certain cell type.

2) ST-ST (Spatial Transcriptomics). Because we already learned the spatial expression representation. The cells 
with similar expression profile can distribute diversed in different tissue samples. Therefore, the learned spatial
distribution of a certain cell type cannot be transferable to a new samples. In this situation, we need to train the cellular distribution across the same sample, and testing the cell composition in the same slide. Using the JSD
metric, we can evaluate the cell type distribution of the fine-tuned model. Hoever, we can also test the performance of our model as cellcontrast, training and testing the slides that close to each other.



### Loading the Lung samples and try to map to the ST data

In [36]:
import scanpy as sc
import matplotlib.pyplot as plt
covid_data = sc.read_h5ad("/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/covid_subsampled.h5ad")

In [34]:
import sys
sys.path.append("/scratch/project_465001027/Spatialformer")
import scanpy as sc
import spatialformer as sp


#configuring the model
model_ckp_path = "/scratch/project_465001027/Spatialformer/output/checkpoints/step=0025000-train_total_loss=-1.3222-val_total_loss=0.0000.ckpt"
tissue = "Lung"
condition = "Disease"

### Calculating the SC pair-wise matrix

In [39]:
covid_data.obs

,age,age_range,anatomical_region,anatomical_region_detailed,batch,dataset,disease,donor,ethnicity,ethnicity_mixed,...,mt_frac,n_counts,n_genes,sample_ID,size_factors,species,tissue,str_batch,batch_id,Gene_Pairs
483188-0-0-1,NaN,nan,nan,nan,1,nan,nan,Donor33,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,HCL,3,"[[IFITM3, CD24], [IFITM3, MTRNR2L12], [ARG1, C..."
P2_8_TCATTTGGTACGAAAT_Krasnow-0-0,46.0,nan,medial,nan,0,Stanford_Krasnow_bioRxivTravaglini,nan,donor_2,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,Krasnow_medial 2,7,"[[HLA-DRA, THBS1], [PGC, MTRNR2L12], [PGC, THB..."
P2_1_GGTGAAGAGAGAACAG_Krasnow-0-0,46.0,nan,distal,nan,0,Stanford_Krasnow_bioRxivTravaglini,nan,donor_2,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,Krasnow_distal 2,5,"[[ACTB, VIM], [ACTB, MTRNR2L12], [ACTB, TCL1A]..."
513429-0-0-1,NaN,nan,nan,nan,1,nan,nan,Donor48,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,HCL,3,"[[XCL2, MTRNR2L12], [VIM, CLU], [VIM, CA1], [V..."
CCTACCACACTCTGTC-1-HCATisStab7659969_Meyer-1-0,NaN,55-60,parenchyma,nan,0,Sanger_Meyer_2019Madissoon,nan,368C,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,Sanger_Meyer_2019Madissoon,13,"[[XCL2, MTRNR2L12], [MGP, MTRNR2L12], [VIM, PG..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20532-0-1-0-1,NaN,nan,nan,nan,1,nan,nan,nan,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,COVID-19 (query),1,"[[VIM, MTRNR2L12], [VIM, CA1], [VIM, PIGR], [V..."
222101-0-0-1,NaN,nan,nan,nan,1,nan,nan,Donor52,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,HCL,3,"[[FCER1G, VIM], [FSCN1, VIM], [HAMP, VIM], [CO..."
31384-0-1-0-1,NaN,nan,nan,nan,1,nan,nan,nan,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,COVID-19 (query),1,"[[HSP90AA1, VIM], [HSP90AA1, MTRNR2L12], [HSP9..."
P2_3_TTGCCGTCAGCTCGCA_Krasnow-0-0,46.0,nan,medial,nan,0,Stanford_Krasnow_bioRxivTravaglini,nan,donor_2,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,Krasnow_medial 2,7,"[[HLA-DRB1, MTRNR2L12], [PTTG1, MTRNR2L12], [H..."


In [40]:
left_cell = ["483188-0-0-1"]
right_cell = ["144-0-1-0-1"]
embeddings, pair_results = sp.tl.embed_data(covid_data, 
                              tissue,
                              condition,
                            model_ckp_path, 
                            batch_size,
                            mode = "pair",
                            left_cell = left_cell,
                            right_cell = right_cell
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


require grad: True
Spatialformer - INFO - Setting the model to the evaluation mode...
Spatialformer - INFO - The model is mapped into cuda
Spatialformer - INFO - Encoding the data into the batch...


1it [00:09,  9.70s/it]


ValueError: Value passed for key 'X_SpaF' is of incorrect shape. Values of obsm must match dimensions (0,) of parent. Value had shape (1, 512) while it should have had (20000,).

In [37]:
%%time
batch_size = 8
embed_adata = sp.tl.embed_data(covid_data, 
                              tissue,
                              condition,
                            model_ckp_path, 
                            batch_size,
                            mode = "single",
                            threshold = 0.7,
                            num_workers = 8
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


require grad: True
Spatialformer - INFO - Setting the model to the evaluation mode...
Spatialformer - INFO - The model is mapped into cuda
Spatialformer - INFO - Encoding the data into the batch...


2500it [08:21,  4.99it/s]


CPU times: user 20min 30s, sys: 2min 36s, total: 23min 6s
Wall time: 8min 34s


In [45]:
covid_data.obsm["X_SpaF"].shape

(20000, 512)

In [46]:
covid_data.obsm["X_SpaF"][0]

array([-2.5661743e-01,  1.3882469e+00, -6.8728268e-01,  3.0857018e-01,
       -1.2993279e+00, -1.0885429e+00,  4.7981945e-01,  4.5349801e-01,
        3.4781182e-01, -9.2655230e-01, -1.1956568e+00, -1.5593736e+00,
       -2.3498498e-01,  4.0409034e-01,  9.3966132e-01, -4.6526816e-01,
       -2.8611848e-01,  1.8866246e+00,  9.7278678e-01,  4.9732146e-01,
       -1.5100441e+00, -1.1831934e+00, -1.4544699e+00,  1.3573568e-01,
       -2.7220598e-01,  1.5575868e-01,  1.0438309e+00,  6.0570419e-01,
       -5.2867508e-01, -1.5038763e-01, -2.1260458e-01, -9.5825684e-01,
       -7.7142173e-01,  8.5592377e-01,  5.5508167e-01, -9.6692108e-02,
       -1.9119350e-02, -8.3392304e-01,  5.0012434e-01,  2.6744676e+00,
        2.1407535e+00, -1.4328111e+00,  1.3173149e+00,  4.1092783e-01,
       -1.5295392e+00, -2.8412659e+00, -6.5228170e-01, -1.9356324e-01,
       -6.2866360e-02, -1.1379837e+00,  2.1342947e-01, -8.5344326e-01,
       -1.5199783e+00,  7.0896953e-01,  2.9173884e-01,  3.8758600e-01,
      

In [72]:
import numpy as np
min_sim = 1
min_sim_cellid = 0
vector1 = covid_data.obsm["X_SpaF"][0]
query_cell_id = covid_data.obs.index[0]
for i in range(20000):
    vector2 = covid_data.obsm["X_SpaF"][i]
    # Calculate the dot product
    dot_product = np.dot(vector1, vector2)

    # Compute the norms of each vector
    norm_vector1 = np.linalg.norm(vector1)
    norm_vector2 = np.linalg.norm(vector2)

    # Calculate cosine similarity
    cosine_similarity = dot_product / (norm_vector1 * norm_vector2)
    if cosine_similarity < min_sim:
        min_sim = cosine_similarity
        min_sim_cellid = covid_data.obs.index[i]
#     print("Cosine Similarity:", cosine_similarity)

min_sim = 0.16247922

but in the pair-wise input, this result turn out to be paired
array([[2.9496021e-05, 9.9997056e-01]], dtype=float32), which means they are paired, and the pair-wise results should not be used to predict the real cases.

### Getting the TOP 5 relative vector for a single cell embeddings

In [85]:
def get_coloc(top=5):
    min_sim = 1
    min_sim_cellid = 0
    all_sim = []
    vector1 = covid_data.obsm["X_SpaF"][2]
    query_cell_id = covid_data.obs.index[0]
    for i in range(covid_data.shape[0]):
        vector2 = covid_data.obsm["X_SpaF"][i]
        # Calculate the dot product
        dot_product = np.dot(vector1, vector2)

        # Compute the norms of each vector
        norm_vector1 = np.linalg.norm(vector1)
        norm_vector2 = np.linalg.norm(vector2)

        # Calculate cosine similarity
        cosine_similarity = dot_product / (norm_vector1 * norm_vector2)
        all_sim.append(cosine_similarity)
    all_sim_array = np.array(all_sim)
    top_indices = np.argsort(all_sim_array)[-top:][::-1]  # Sort indices and take the last 5, then reverse them
    cellids = covid_data.obs.index[top_indices]
    celltypes = covid_data.obs["celltype"][top_indices]
    # Get the top 5 maximum values
    top_values = all_sim_array[top_indices]
    return top_indices, cellids, celltypes, top_values

In [86]:
get_coloc(top=50)

(array([    2, 13049,  1945, 14401,  3992,  7602, 14488,  3647, 15073,
         6005, 19524, 16912,  8408, 18699, 18927, 16485, 10855, 14284,
         5595,  5639,  5386, 18638, 12714,  4483,  9291, 18597, 14233,
         7089, 17377, 14330,  4911, 11962, 16696, 19835,  6963, 12524,
        13001, 17940, 10154, 18567,  3982,    31,  1981, 14677,  2566,
         5988,  4279,  2562,   532,  8629]),
 Index(['P2_1_GGTGAAGAGAGAACAG_Krasnow-0-0',
        'CGGGTCACATTGAGCT-1-HCATisStab7509734_Meyer-1-0',
        'P3_6_AACCATGGTTAGAACA_Krasnow-0-0',
        'CGGTTAAGTCGCGTGT-1-HCATisStab7747198_Meyer-1-0',
        'CATGCCTAGGACTGGT-1-HCATisStab7509736_Meyer-1-0',
        'P3_6_AGTGTCAAGGTAGCCA_Krasnow-0-0',
        'TTAACTCTCCTAAGTG_Donor_06_Misharin-1-0',
        'P3_6_GATTCAGAGCCAGAAC_Krasnow-0-0',
        'GCGCCAAGTCTTTCAT-1-HCATisStab7587202_Meyer-1-0',
        'P2_4_CGTCTACAGCCAGAAC_Krasnow-0-0',
        'P3_6_GTGGGTCAGTGCTGCC_Krasnow-0-0',
        'AGTGTCAAGTTCCACA_Donor_06_Misharin-1-0'

In [81]:
covid_data.obs["celltype"]

483188-0-0-1                                         Monocytes
P2_8_TCATTTGGTACGAAAT_Krasnow-0-0                   Mast cells
P2_1_GGTGAAGAGAGAACAG_Krasnow-0-0                          AT2
513429-0-0-1                                            T cell
CCTACCACACTCTGTC-1-HCATisStab7659969_Meyer-1-0    CD4+ T cells
                                                      ...     
20532-0-1-0-1                                      Macrophages
222101-0-0-1                                       Macrophages
31384-0-1-0-1                                      Macrophages
P2_3_TTGCCGTCAGCTCGCA_Krasnow-0-0                 CD4+ T cells
P1_2_TTGCGTCAGGCTCTTA_Krasnow-0-0                  Macrophages
Name: celltype, Length: 20000, dtype: category
Categories (39, object): ['AT1', 'AT2', 'B cell', 'CCR7+ T', ..., 'Tregs', 'innate T', 'mDC', 'pDC']